# Inspire Hand Control Notebook
This notebook demonstrates how to control the Inspire FTP Dexterity Hand using the DDS SDK.

In [ ]:
import time
import sys
import numpy as np

from unitree_sdk2py.core.channel import ChannelPublisher, ChannelFactoryInitialize
from inspire_sdkpy import inspire_hand_defaut, inspire_dds

## Initialize Channel
Initialize the DDS channel factory. If a network interface is needed, pass it as an argument (e.g., 'eth0').

In [ ]:
ChannelFactoryInitialize(0)

## Create Publishers
Create publishers for the left and right hands.

In [ ]:
pubr = ChannelPublisher("rt/inspire_hand/ctrl/r", inspire_dds.inspire_hand_ctrl)
pubr.Init()

publ = ChannelPublisher("rt/inspire_hand/ctrl/l", inspire_dds.inspire_hand_ctrl)
publ.Init()

## Control Logic
Define the command object and set initial values.

In [ ]:
cmd = inspire_hand_defaut.get_inspire_hand_ctrl()

# Set initial angle (0-1000 range per joint)
cmd.angle_set = [0, 0, 0, 0, 1000, 1000]
cmd.mode = 0b0001 # Angle control mode

## Send Command
Send a command to both hands.

In [ ]:
publ.Write(cmd)
pubr.Write(cmd)
print("Command sent!")

## Dynamic Movement Example
Move the fingers in a loop.

In [ ]:
try:
    short_value = 1000
    for cnd in range(100): # Run for a specific iterations
        if (cnd + 1) % 10 == 0:
            short_value = 1000 - short_value

        values_to_write = [short_value] * 6
        values_to_write[-1] = 1000 - values_to_write[-1]
        values_to_write[-2] = 1000 - values_to_write[-2]

        value_to_write_np = np.array(values_to_write)
        value_to_write_np = np.clip(value_to_write_np, 200, 800)

        cmd.angle_set = value_to_write_np.tolist()
        cmd.mode = 0b0001

        if publ.Write(cmd) and pubr.Write(cmd):
            # print("Publish success.")
            pass
        else:
            print("Waiting for subscriber...")

        time.sleep(0.1)
except KeyboardInterrupt:
    print("Stopped by user")